## Importing Brain LLM

In [1]:
%load_ext autoreload
%autoreload 2
###? Auto Reloads if python files are edited , no kernel restart needed.

In [2]:
import os

os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

In [3]:
import os
import dotenv
dotenv.load_dotenv()

True

In [4]:
from Agent import Client

In [5]:
API = os.getenv("GROQ_API_KEY")
Model = os.getenv("MODEL")

In [ ]:
from sparse_ai.core.enums import Providers

client = Client(api_key=API , model=Model , provider=Providers.GROQ)

In [7]:
def check_emails():
    """Get List of user's emails."""
    return {
        "messages": [
            {
                "id": "18c7a9f2b4d12345",
                "thread_id": "18c7a9f2b4d12345",
                "from": "Rahul Sharma <rahul@example.com>",
                "to": "priyansh@example.com",
                "subject": "Project meeting tomorrow",
                "date": "2026-09-19T14:32:00Z",
                "snippet": "Hey Priyansh, are we still meeting tomorrow at 11 AM?",
                "labels": ["INBOX", "UNREAD"]
            },
            {
                "id": "18c7a8e1c3f67890",
                "thread_id": "18c7a8e1c3f67890",
                "from": "GitHub <noreply@github.com>",
                "to": "priyansh@example.com",
                "subject": "Security alert",
                "date": "2026-09-19T12:15:00Z",
                "snippet": "A new sign-in to your GitHub account was detected...",
                "labels": ["INBOX", "IMPORTANT", "UNREAD"]
            },
            {
                "id": "18c7a71234abcd56",
                "thread_id": "18c7a71234abcd56",
                "from": "Amazon <order-update@amazon.in>",
                "to": "priyansh@example.com",
                "subject": "Your Amazon order has shipped",
                "date": "2026-09-19T09:41:00Z",
                "snippet": "Your package has shipped and is expected to arrive...",
                "labels": ["INBOX"]
            }
        ],
        "total": 3
    }

In [ ]:
from sparse_ai.features.tools import approval_request

In [9]:
@approval_request()
def schedule_task(datetime , task):
    """Schedule a Task reminder."""
    return {'status': 'Reminder set.'}

In [10]:
def search_internet(query):
    """Search intrernet to get information by writing query."""

    return {
        'result':{
            'query':query,
            'wibsite': {'itch.io', 'f95zone', 'paterion'},
            'content': "Latest version of Eternum AVN is v0.10"
        }
    }

In [11]:
from Agent import Tool

In [12]:
tools = [
    Tool(func=search_internet),
    Tool(schedule_task),
    Tool(check_emails)
]

In [13]:
Tool.serialize_tool(tools)

[{'name': 'search_internet',
  'description': 'Search intrernet to get information by writing query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string'}},
   'required': ['query']}},
 {'name': 'schedule_task',
  'description': 'Schedule a Task reminder.\n\nBefore execution, provide an approval explanation.',
  'parameters': {'type': 'object',
   'properties': {'datetime': {'type': 'string'},
    'task': {'type': 'string'},
    '__approval_request__': {'type': 'string',
     'description': 'Briefly state what result or change this call will produce.'}},
   'required': ['datetime', 'task', '__approval_request__']}},
 {'name': 'check_emails',
  'description': "Get List of user's emails.",
  'parameters': {'type': 'object', 'properties': {}, 'required': []}}]

In [14]:
from Agent import Message

In [15]:
messages = [
    Message.system("You are a helpfull AI assistant designed to help user with their tasks."
                   "You are provided with tools that you can use to complete a task."
                   "Be respectful and polite.")
]

In [16]:
from Agent import State

In [17]:
state = State(messages=messages)

In [34]:
from Agent import Graph 

In [ ]:
from sparse_ai.features.graph import Node, START , END

In [36]:
graph = Graph()

In [37]:
def chat_router(state):
    if state.tool_calls:
        return 'tool'
    else:
        return END

In [38]:
graph.graph_builder(
    nodes={
        'chat': Node.llm_call(),
        'tool': Node.tool_exec(),
        'approval': Node.approval_node(on_approved='tool', on_rejected='chat'),
    },
    edges=[
        (START , 'chat'),
        ('tool', 'chat')
    ],
    routers=[
        ('chat', chat_router,  {'If Tool call': 'tool' , 'no tool call':'END'} )
    ]
)

Graph(entry='chat')
  nodes: ['chat', 'tool', 'approval']
  approval -> <router:_approval_router>
  tool -> chat
  chat -> <router:chat_router>

In [43]:
graph.display_graph(browser_view=True)

'digraph AgentGraph {\n    rankdir=TB;\n    splines=polyline;\n    nodesep="0.8";\n    ranksep="1.0";\n    concentrate=false;\n    bgcolor="transparent";\n    node [fontname="Segoe UI, Helvetica, sans-serif", fontsize=12, style="filled,rounded", margin="0.2,0.1", penwidth=1.6];\n    edge [fontname="Segoe UI, Helvetica, sans-serif", fontsize=10, color="#455A64", penwidth=1.3, arrowsize=0.85];\n    START [label="START", shape=box, fillcolor="#4CAF50", color="#2E7D32", fontcolor="#FFFFFF", style="filled,rounded"];\n    END [label="END", shape=box, fillcolor="#EF5350", color="#C62828", fontcolor="#FFFFFF", style="filled,rounded"];\n    chat [label="  Chat  ", shape=diamond, style=filled, fillcolor="#FFF3E0", color="#E65100", fontcolor="#BF360C", penwidth=2.6];\n    tool [label="Tool", shape=box, style="filled,rounded", fillcolor="#E3F2FD", color="#1565C0", fontcolor="#1A1A1A", penwidth=1.6];\n    approval [label="  Approval  ", shape=diamond, style=filled, fillcolor="#FFF3E0", color="#E651

In [44]:
from Agent import Agent

In [45]:
agent = Agent(client=client ,tools=tools,graph=graph ,messages=messages, state= state , auto_handle_interrupts=True)

In [46]:
response = await agent.run("Wassup!! I am Priyansh!!")
print(response)

DEBUG:GroqAdapter:[call :1], sending 2 messages
DEBUG:Agent.core.logging:SERIALIZED TOOLS: [
  {
    "type": "function",
    "function": {
      "name": "search_internet",
      "description": "Search intrernet to get information by writing query.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string"
          }
        },
        "required": [
          "query"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "schedule_task",
      "description": "Schedule a Task reminder.\n\nBefore execution, provide an approval explanation.",
      "parameters": {
        "type": "object",
        "properties": {
          "datetime": {
            "type": "string"
          },
          "task": {
            "type": "string"
          },
          "__approval_request__": {
            "type": "string",
            "description": "Briefly state what result or change this call will produce."


=================================AGENT RUNNING=======================================
================================= STATE =================================================
 State(messages=[Message(role='system', content='You are a helpfull AI assistant designed to help user with their tasks.You are provided with tools that you can use to complete a task.Be respectful and polite.', raw=None), Message(role='user', content='Wassup!! I am Priyansh!!', raw=None)], last_response=None, tool_calls=None, tool_results=[], current_node_name=None, status=None, run_id=None, executed_nodes=[], node_retries={}, agent_retries=3, interrupt=None, approval_response=None, approval_result=None, approval_notes=None, llm_approval_requests={}, custom_fields={})



MESSAGE RECIVED TO ADAPTER:
 [{'role': 'system', 'content': 'You are a helpfull AI assistant designed to help user with their tasks.You are provided with tools that you can use to complete a task.Be respectful and polite.'}, {'role': 'user', 'co

DEBUG:httpcore2.connection:connect_tcp.started host='api.groq.com' port=443 local_address=None timeout=5.0 socket_options=None
DEBUG:httpcore2.connection:connect_tcp.complete return_value=<httpcore2._backends.anyio.AnyIOStream object at 0x0000027469375550>
DEBUG:httpcore2.connection:start_tls.started ssl_context=<truststore._api.SSLContext object at 0x000002746571E530> server_hostname='api.groq.com' timeout=5.0
DEBUG:httpcore2.connection:start_tls.complete return_value=<httpcore2._backends.anyio.AnyIOStream object at 0x0000027469384B90>
DEBUG:httpcore2.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore2.http11:send_request_headers.complete
DEBUG:httpcore2.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore2.http11:send_request_body.complete
DEBUG:httpcore2.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore2.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed,




###############################################################ADAPTER GENERATED RESPONSE:
 ChatCompletionMessage(content='Hey Priyansh! 👋 How can I help you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User says "Wassup!! I am Priyansh!!". They are greeting. We should respond politely. No tool needed.')
Hey Priyansh! 👋 How can I help you today?


In [47]:
response = await agent.run("What is the latest version of the game 'Eternum' ?")
print(response)

DEBUG:GroqAdapter:[call :2], sending 4 messages
DEBUG:Agent.core.logging:SERIALIZED TOOLS: [
  {
    "type": "function",
    "function": {
      "name": "search_internet",
      "description": "Search intrernet to get information by writing query.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string"
          }
        },
        "required": [
          "query"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "schedule_task",
      "description": "Schedule a Task reminder.\n\nBefore execution, provide an approval explanation.",
      "parameters": {
        "type": "object",
        "properties": {
          "datetime": {
            "type": "string"
          },
          "task": {
            "type": "string"
          },
          "__approval_request__": {
            "type": "string",
            "description": "Briefly state what result or change this call will produce."


=================================AGENT RUNNING=======================================
================================= STATE =================================================
 State(messages=[Message(role='system', content='You are a helpfull AI assistant designed to help user with their tasks.You are provided with tools that you can use to complete a task.Be respectful and polite.', raw=None), Message(role='user', content='Wassup!! I am Priyansh!!', raw=None), Message(role='assistant', content='Hey Priyansh! 👋 How can I help you today?', raw=None), Message(role='user', content="What is the latest version of the game 'Eternum' ?", raw=None)], last_response=LLMResponse(text='Hey Priyansh! 👋 How can I help you today?', tool_calls=None), tool_calls=None, tool_results=[], current_node_name='chat', status=<StateEvent.DONE: 'done'>, run_id=None, executed_nodes=['chat'], node_retries={}, agent_retries=3, interrupt=None, approval_response=None, approval_result=None, approval_notes=None, llm_a

DEBUG:httpcore2.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 23 Sep 2026 15:24:44 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'bom'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'8000'), (b'x-ratelimit-remaining-requests', b'998'), (b'x-ratelimit-remaining-tokens', b'7486'), (b'x-ratelimit-reset-requests', b'2m52.8s'), (b'x-ratelimit-reset-tokens', b'3.855s'), (b'x-request-id', b'req_01m37dx0w9ejdaxqqhad8k3eqk'), (b'via', b'1.1 google'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=15552000'), (b'Content-Encoding', b'gzip'), (b'CF-RAY', b'a3fa8ed85fdf9a83-NAG'), (b'alt-svc', b'h3=":443"; ma=86400')])
INFO:httpx2:HTTP Request: POST https://api.groq.com/openai/




###############################################################ADAPTER GENERATED RESPONSE:
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_a1217e62-33e5-463b-bd96-8c6f27803657', function=Function(arguments='{"query":"latest version of the game Eternum"}', name='search_internet'), type='function')], reasoning="We need to find latest version of game 'Eternum'. Likely a game. We need to search internet. Use search_internet.")



MESSAGE RECIVED TO ADAPTER:
 [{'role': 'system', 'content': 'You are a helpfull AI assistant designed to help user with their tasks.You are provided with tools that you can use to complete a task.Be respectful and polite.'}, {'role': 'user', 'content': 'Wassup!! I am Priyansh!!'}, {'role': 'assistant', 'content': 'Hey Priyansh! 👋 How can I help you today?'}, {'role': 'user', 'content': "What is the latest version of the game 'Eternum' 

DEBUG:httpcore2.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 23 Sep 2026 15:24:45 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'bom'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'8000'), (b'x-ratelimit-remaining-requests', b'997'), (b'x-ratelimit-remaining-tokens', b'7110'), (b'x-ratelimit-reset-requests', b'4m19.2s'), (b'x-ratelimit-reset-tokens', b'6.675s'), (b'x-request-id', b'req_01m37dx1breh9b9zdpv4qv26kz'), (b'Strict-Transport-Security', b'max-age=15552000'), (b'via', b'1.1 google'), (b'cf-cache-status', b'DYNAMIC'), (b'Content-Encoding', b'gzip'), (b'CF-RAY', b'a3fa8edbbbc69a83-NAG'), (b'alt-svc', b'h3=":443"; ma=86400')])
INFO:httpx2:HTTP Request: POST https://api.groq.com/openai/




###############################################################ADAPTER GENERATED RESPONSE:
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_b115d0c3-5d46-4c9e-9926-c5decfc935c9', function=Function(arguments='{"query":"Eternum game version 0.10"}', name='search_internet'), type='function')], reasoning='The search result says "Latest version of Eternum AVN is v0.10". But we need to confirm. Let\'s search again.')



MESSAGE RECIVED TO ADAPTER:
 [{'role': 'system', 'content': 'You are a helpfull AI assistant designed to help user with their tasks.You are provided with tools that you can use to complete a task.Be respectful and polite.'}, {'role': 'user', 'content': 'Wassup!! I am Priyansh!!'}, {'role': 'assistant', 'content': 'Hey Priyansh! 👋 How can I help you today?'}, {'role': 'user', 'content': "What is the latest version of the game 'Eternum' ?"}, {'role':

DEBUG:httpcore2.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 23 Sep 2026 15:24:45 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'bom'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'8000'), (b'x-ratelimit-remaining-requests', b'996'), (b'x-ratelimit-remaining-tokens', b'6591'), (b'x-ratelimit-reset-requests', b'5m45.6s'), (b'x-ratelimit-reset-tokens', b'10.567s'), (b'x-request-id', b'req_01m37dx1wvee3am92a8bja3yza'), (b'via', b'1.1 google'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=15552000'), (b'Content-Encoding', b'gzip'), (b'CF-RAY', b'a3fa8edf2fdd9a83-NAG'), (b'alt-svc', b'h3=":443"; ma=86400')])
INFO:httpx2:HTTP Request: POST https://api.groq.com/openai




###############################################################ADAPTER GENERATED RESPONSE:
 ChatCompletionMessage(content='Hey Priyansh! The most recent release of **Eternum** (the “Eternum AVN” version) is **v0.10**. If you’re looking to download or update, you can usually find it on the game’s official page or on platforms like itch.io. Let me know if you need help installing it or anything else!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='It seems the latest version is v0.10. Provide answer.')
Hey Priyansh! The most recent release of **Eternum** (the “Eternum AVN” version) is **v0.10**. If you’re looking to download or update, you can usually find it on the game’s official page or on platforms like itch.io. Let me know if you need help installing it or anything else!


In [48]:
from pprint import pprint
pprint(repr(state))

("State(messages=[Message(role='system', content='You are a helpfull AI "
 'assistant designed to help user with their tasks.You are provided with tools '
 "that you can use to complete a task.Be respectful and polite.', raw=None), "
 "Message(role='user', content='Wassup!! I am Priyansh!!', raw=None), "
 "Message(role='assistant', content='Hey Priyansh! 👋 How can I help you "
 'today?\', raw=None), Message(role=\'user\', content="What is the latest '
 'version of the game \'Eternum\' ?", raw=None), Message(role=\'assistant\', '
 "content=None, raw={'role': 'assistant', 'tool_calls': [{'id': "
 "'fc_a1217e62-33e5-463b-bd96-8c6f27803657', 'function': {'arguments': "
 '\'{"query":"latest version of the game Eternum"}\', \'name\': '
 '\'search_internet\'}, \'type\': \'function\'}], \'reasoning\': "We need to '
 "find latest version of game 'Eternum'. Likely a game. We need to search "
 'internet. Use search_internet."}), Message(role=\'tool\', content=None, '
 "raw={'name': 'search_intern